# NumPy: Real-World NumPy

### Part 4 of 5 in the NumPy series

**What you'll learn:**
1. Loading and saving data — text files, binary, NumPy's `.npy` format
2. Handling missing values
3. Working with dates and time series
4. Performance tips — when NumPy is slow and how to fix it
5. Memory management — copies vs views (revisited)
6. Common real-world patterns from actual codebases
7. Random number generation in depth


In [1]:
import numpy as np
import os
np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## Loading and Saving Data

You'll constantly need to get data in and out of NumPy. Here are the main ways.

### Text files (CSV, etc.) with `loadtxt` and `genfromtxt`

`np.loadtxt` is simple but strict — no missing values allowed. `np.genfromtxt` is more forgiving and handles messy data.

Let's create a small CSV to work with.

In [3]:
# Create a sample CSV
sample_csv = '''10,20,30
40,50,60
70,80,90
100,110,120'''

with open('sample.csv', 'w') as f:
    f.write(sample_csv)

# Load it
data = np.loadtxt('sample.csv', delimiter=',')
print("Loaded data:")
print(data)
print(f"Shape: {data.shape}")
print(f"Dtype: {data.dtype}")

Loaded data:
[[ 10.  20.  30.]
 [ 40.  50.  60.]
 [ 70.  80.  90.]
 [100. 110. 120.]]
Shape: (4, 3)
Dtype: float64


In [4]:
# CSV with a header and missing values
messy_csv = '''age,height,weight
25,170,65
30,,72
,165,55
45,178,80'''

with open('messy.csv', 'w') as f:
    f.write(messy_csv)

# genfromtxt handles the missing values
data = np.genfromtxt('messy.csv', delimiter=',', skip_header=1)
print("Data with missing values:")
print(data)
print("\nNotice the 'nan' values where data was missing")

Data with missing values:
[[ 25. 170.  65.]
 [ 30.  nan  72.]
 [ nan 165.  55.]
 [ 45. 178.  80.]]

Notice the 'nan' values where data was missing


### NumPy's native format: `.npy` and `.npz`

For NumPy-to-NumPy workflows, the native `.npy` format is fastest and preserves dtype/shape perfectly.

In [5]:
# Save a single array
arr = np.arange(100).reshape(10, 10)
np.save('myarray.npy', arr)

# Load it back
loaded = np.load('myarray.npy')
print("Loaded shape:", loaded.shape)
print("First row:   ", loaded[0])
print("Match original?", np.array_equal(arr, loaded))

Loaded shape: (10, 10)
First row:    [0 1 2 3 4 5 6 7 8 9]
Match original? True


In [6]:
# Save MULTIPLE arrays in one file with .npz
a = np.array([1, 2, 3])
b = np.array([[10, 20], [30, 40]])
c = np.linspace(0, 1, 5)

np.savez('multi.npz', alpha=a, beta=b, gamma=c)

# Load and access by name
archive = np.load('multi.npz')
print("Keys:", list(archive.keys()))
print("alpha:", archive['alpha'])
print("beta:")
print(archive['beta'])
print("gamma:", archive['gamma'])

Keys: ['alpha', 'beta', 'gamma']
alpha: [1 2 3]
beta:
[[10 20]
 [30 40]]
gamma: [0.   0.25 0.5  0.75 1.  ]


**When to use each format:**

| Format | Use case |
|---|---|
| `.csv` (loadtxt/savetxt) | Sharing with non-Python tools (Excel, R, etc.) |
| `.npy` (save/load) | NumPy-only workflows — fastest, smallest |
| `.npz` (savez) | Multiple arrays in one file |
| `.npz` (savez_compressed) | Saving disk space (slower to load) |
| pickle / parquet / HDF5 | Production pipelines — outside the scope of NumPy itself |

## Handling Missing Values (NaN)

NumPy represents missing numeric data with `np.nan` (Not a Number).

In [7]:
data = np.array([1.0, 2.0, np.nan, 4.0, np.nan, 6.0])
print("Data:", data)

# Most operations propagate NaN — they "infect" everything
print("\nmean:", data.mean())           # nan
print("sum: ", data.sum())               # nan

# Use the nan-aware versions to ignore them
print("\nnp.nanmean:", np.nanmean(data))
print("np.nansum: ", np.nansum(data))
print("np.nanstd: ", np.nanstd(data))
print("np.nanmin: ", np.nanmin(data))
print("np.nanmax: ", np.nanmax(data))

Data: [ 1.  2. nan  4. nan  6.]

mean: nan
sum:  nan

np.nanmean: 3.25
np.nansum:  13.0
np.nanstd:  1.920286436967152
np.nanmin:  1.0
np.nanmax:  6.0


In [8]:
# Detect NaN values
data = np.array([1.0, 2.0, np.nan, 4.0, np.nan, 6.0])

#  NaN is never equal to anything — even itself!
print("np.nan == np.nan:", np.nan == np.nan)   # False!

# Use np.isnan() instead
print("np.isnan(data):", np.isnan(data))

# Count missing
print("\nNumber of NaNs:", np.isnan(data).sum())
print("Number of valid:", (~np.isnan(data)).sum())

np.nan == np.nan: False
np.isnan(data): [False False  True False  True False]

Number of NaNs: 2
Number of valid: 4


In [9]:
# Replacing NaN values
data = np.array([1.0, 2.0, np.nan, 4.0, np.nan, 6.0])

# Option 1: replace with 0
clean = np.where(np.isnan(data), 0, data)
print("Replaced with 0:   ", clean)

# Option 2: replace with the mean
mean = np.nanmean(data)
clean = np.where(np.isnan(data), mean, data)
print("Replaced with mean:", clean)

# Option 3: drop the NaN values entirely
no_nan = data[~np.isnan(data)]
print("Dropped NaNs:      ", no_nan)

Replaced with 0:    [1. 2. 0. 4. 0. 6.]
Replaced with mean: [1.   2.   3.25 4.   3.25 6.  ]
Dropped NaNs:       [1. 2. 4. 6.]


## Dates and Time Series

NumPy has `datetime64` for dates and `timedelta64` for durations. Quite useful for time series before you need pandas.

In [10]:
# Create dates
today = np.datetime64('2026-06-15')
print("Today:", today)

# Date arithmetic with timedelta
one_week = np.timedelta64(7, 'D')
print("Next week:", today + one_week)
print("100 days from today:", today + np.timedelta64(100, 'D'))

# Array of dates
dates = np.arange('2026-06-01', '2026-06-10', dtype='datetime64[D]')
print("\nA week of dates:")
print(dates)

Today: 2026-06-15
Next week: 2026-06-22
100 days from today: 2026-09-23

A week of dates:
['2026-06-01' '2026-06-02' '2026-06-03' '2026-06-04' '2026-06-05'
 '2026-06-06' '2026-06-07' '2026-06-08' '2026-06-09']


In [11]:
# Extract year, month, day
dates = np.array(['2026-01-15', '2026-06-20', '2026-12-31'], dtype='datetime64[D]')

# Cast to different units
years  = dates.astype('datetime64[Y]')   # truncated to year
months = dates.astype('datetime64[M]')   # truncated to month

print("Original:", dates)
print("Years:   ", years)
print("Months:  ", months)

# Days between two dates
delta = np.datetime64('2026-12-31') - np.datetime64('2026-06-15')
print(f"\nDays from June 15 to year end: {delta.astype(int)}")

Original: ['2026-01-15' '2026-06-20' '2026-12-31']
Years:    ['2026' '2026' '2026']
Months:   ['2026-01' '2026-06' '2026-12']

Days from June 15 to year end: 199


## Performance — When NumPy is Slow

NumPy is fast for vectorized operations. It is SLOW when you write Python loops over arrays. Here's how to spot and fix this.

In [ ]:
import time

# Slow: Python loop
def slow_square(arr):
    result = np.zeros_like(arr)
    for i in range(len(arr)):
        result[i] = arr[i] ** 2
    return result

# Fast: vectorized
def fast_square(arr):
    return arr ** 2

data = np.arange(1_000_000)

# Time the slow version
start = time.time()
slow_result = slow_square(data)
t_slow = time.time() - start

# Time the fast version
start = time.time()
fast_result = fast_square(data)
t_fast = time.time() - start

print(f"Slow (Python loop):  {t_slow:.4f}s")
print(f"Fast (vectorized):   {t_fast:.4f}s")
print(f"Speedup:             {t_slow/t_fast:.1f}x")

### For numpy performance: AVOID Python loops over array elements.

If you find yourself writing `for i in range(len(arr)):`, stop and ask: can this be vectorized?

### Common loop replacements

| Loop pattern | Vectorized alternative |
|---|---|
| Element-wise math | Use operators directly: `a + b`, `a ** 2` |
| Conditional updates | `np.where(condition, x, y)` |
| Cumulative sum | `np.cumsum(a)` |
| Apply function to each element | Use a ufunc (often `np.vectorize` is NOT faster, surprisingly) |
| Pairwise operations | Use broadcasting |

In [12]:
# Example: cumulative sum without a loop
arr = np.array([1, 2, 3, 4, 5])
print("Cumulative sum:", np.cumsum(arr))
print("Cumulative product:", np.cumprod(arr))

Cumulative sum: [ 1  3  6 10 15]
Cumulative product: [  1   2   6  24 120]


## Memory — Copies vs Views 

When does an operation copy data, and when does it return a view?

### Views (no copy)

These don't copy memory, they're just different ways of looking at the same data:
- Basic slicing: `a[1:5]`, `a[:, 0]`
- Reshape (sometimes): `a.reshape(...)`
- Transpose: `a.T`
- `ravel()` (sometimes)

In [13]:
a = np.arange(10)
b = a[2:7]  # view

# Check: do they share memory?
print("Shares memory:", np.shares_memory(a, b)) 
print("b.base is a:  ", b.base is a)   # True for views

Shares memory: True
b.base is a:   True


### Copies (new memory)

These DO copy memory:
- Fancy indexing: `a[[0, 2, 4]]`
- Boolean masking: `a[a > 5]`
- Explicit `.copy()`
- Most ufunc results: `a + b`

In [14]:
a = np.arange(10)
b = a[[0, 2, 4]]  # fancy indexing — copies

print("Shares memory:", np.shares_memory(a, b))   # False
print("b.base:       ", b.base)                   # None for copies

Shares memory: False
b.base:        None


In [15]:
# Why this matters: modifying views modifies the original
arr = np.arange(20).reshape(4, 5)
print("Original:")
print(arr)

# Slice gives a view
row = arr[1]
row[:] = 0     # modify all of "row"

print("\nAfter setting row 1 to 0:")
print(arr)   # ← original changed!

Original:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

After setting row 1 to 0:
[[ 0  1  2  3  4]
 [ 0  0  0  0  0]
 [10 11 12 13 14]
 [15 16 17 18 19]]


In [16]:
# To avoid this — use .copy()
arr = np.arange(20).reshape(4, 5)
row = arr[1].copy()
row[:] = 999

print("Original (unchanged):")
print(arr)
print("\nIndependent copy:")
print(row)

Original (unchanged):
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

Independent copy:
[999 999 999 999 999]


## Random Number Generation 

You'll use random number generation constantly for ML — initializing weights, shuffling data, sampling, simulations.

### The modern API: `np.random.default_rng()`

NumPy 1.17+ introduced a new random API. It's the recommended way going forward.

In [19]:
# Old API (still works, but legacy)
np.random.seed(42)
old_way = np.random.rand(3)
print("Old API:", old_way)

# Modern API — preferred
rng = np.random.default_rng(seed=42)
new_way = rng.random(3)
print("New API:", new_way)

Old API: [0.37454012 0.95071431 0.73199394]
New API: [0.77395605 0.43887844 0.85859792]


In [21]:
rng = np.random.default_rng(seed=42)

# Common distributions
print("Uniform [0, 1):       ", rng.random(5).round(4))
print("Uniform [10, 20]:     ", rng.uniform(10, 20, 5).round(2))
print("Random integers 0-99: ", rng.integers(0, 100, 5))
print("Normal mean=0, std=1: ", rng.normal(0, 1, 5).round(4))
print("Normal mean=50, std=5:", rng.normal(50, 5, 5).round(2))

Uniform [0, 1):        [0.774  0.4389 0.8586 0.6974 0.0942]
Uniform [10, 20]:      [19.76 17.61 17.86 11.28 14.5 ]
Random integers 0-99:  [50 37 18 92 78]
Normal mean=0, std=1:  [ 1.1272  0.4675 -0.8593  0.3688 -0.9589]
Normal mean=50, std=5: [54.39 49.75 49.08 46.6  56.11]


In [22]:
# Sampling without replacement (no duplicates)
rng = np.random.default_rng(42)
data = np.arange(10)

sample = rng.choice(data, size=5, replace=False)
print("Sample (no replace):", sample)

# Sampling with replacement (duplicates possible)
sample = rng.choice(data, size=5, replace=True)
print("Sample (replace):   ", sample)

# Weighted sampling
items = np.array(['rare', 'common', 'frequent'])
weights = [0.05, 0.25, 0.70]
sample = rng.choice(items, size=10, p=weights)
print("Weighted sample:    ", sample)

Sample (no replace): [5 3 7 0 4]
Sample (replace):    [0 5 9 7 7]
Weighted sample:     ['frequent' 'common' 'frequent' 'frequent' 'frequent' 'frequent'
 'frequent' 'frequent' 'common' 'frequent']


In [23]:
# Shuffling an array
rng = np.random.default_rng(42)
data = np.arange(10)

# rng.shuffle MODIFIES IN PLACE
copy = data.copy()
rng.shuffle(copy)
print("Shuffled (in place):", copy)

# For a non-mutating version, use permutation
permuted = rng.permutation(data)
print("Permuted (new arr): ", permuted)
print("Original unchanged: ", data)

Shuffled (in place): [5 6 0 7 3 2 4 9 1 8]
Permuted (new arr):  [4 8 2 6 5 9 7 3 0 1]
Original unchanged:  [0 1 2 3 4 5 6 7 8 9]


## Real-World Patterns 

These are idioms you'll see constantly in ML and data code.

### Pattern 1: Train/test split (no sklearn needed)

In [24]:
# Generate some fake features and labels
rng = np.random.default_rng(0)
n = 100
X = rng.normal(0, 1, (n, 4))     # 100 samples, 4 features
y = rng.integers(0, 2, n)        # binary labels

# Shuffle indices
indices = rng.permutation(n)

# Split 80/20
split = int(0.8 * n)
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

X_train shape: (80, 4), X_test shape: (20, 4)
y_train shape: (80,), y_test shape: (20,)


### Pattern 2: Normalization (standard scaling)

Subtract the mean, divide by the standard deviation. Essential for many ML algorithms.

In [25]:
# 10 samples, 3 features
data = np.array([[170, 65, 25],
                 [180, 80, 30],
                 [160, 55, 22],
                 [175, 72, 28],
                 [165, 60, 24]])

# Per-column mean and std
mean = data.mean(axis=0)
std  = data.std(axis=0)

# Standardize: each column now has mean=0, std=1
standardized = (data - mean) / std

print("Mean of standardized (should be ~0):", standardized.mean(axis=0).round(6))
print("Std of standardized  (should be ~1):", standardized.std(axis=0).round(6))

Mean of standardized (should be ~0): [ 0. -0. -0.]
Std of standardized  (should be ~1): [1. 1. 1.]


### Pattern 3: One-hot encoding

In [26]:
# Categories represented as integers
labels = np.array([0, 2, 1, 1, 0, 2])
n_classes = 3

# Identity matrix indexing — clever one-hot trick
one_hot = np.eye(n_classes)[labels]
print("Original labels:", labels)
print("One-hot encoded:")
print(one_hot)

Original labels: [0 2 1 1 0 2]
One-hot encoded:
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]


### Pattern 4: Compute distances between all pairs of points

In [27]:
# 5 points in 2D
points = np.array([[0, 0],
                   [1, 1],
                   [2, 0],
                   [0, 3],
                   [4, 4]])

# Pairwise distance matrix using broadcasting
# points shape: (5, 2)
# Add a new axis to make broadcasting work: (5, 1, 2) - (1, 5, 2) → (5, 5, 2)
diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
distances = np.sqrt((diff ** 2).sum(axis=2))

print("Distance matrix:")
print(distances.round(2))

Distance matrix:
[[0.   1.41 2.   3.   5.66]
 [1.41 0.   1.41 2.24 4.24]
 [2.   1.41 0.   3.61 4.47]
 [3.   2.24 3.61 0.   4.12]
 [5.66 4.24 4.47 4.12 0.  ]]


### Pattern 5: Running average with `np.convolve` or `cumsum`

In [28]:
# Moving average — smoothing noisy data
noisy = np.array([1, 5, 2, 8, 3, 7, 4, 9, 5, 6])

# Method 1: convolve with a uniform kernel
window = 3
kernel = np.ones(window) / window
smooth = np.convolve(noisy, kernel, mode='valid')
print("Noisy:  ", noisy)
print(f"Smoothed (window={window}):", smooth.round(2))

Noisy:   [1 5 2 8 3 7 4 9 5 6]
Smoothed (window=3): [2.67 5.   4.33 6.   4.67 6.67 6.   6.67]


## Practice Exercises

### Exercise 1
Create an array `data = np.array([1.0, 2.0, np.nan, 4.0, 5.0, np.nan, 7.0])`. Replace the NaN values with the median of the non-NaN values.

### Exercise 2
Save the array `np.arange(100).reshape(10, 10)` to disk as a `.npy` file. Load it back and verify it's the same.

### Exercise 3
Given `np.random.seed(0)` and a 100x5 array of random floats, find:
- The standardized (mean=0, std=1) version of each column
- The two columns with the highest correlation

### Exercise 4 (performance)
Compare the time for these two ways to compute `y = sin(x) + cos(x) * 2` for `x = np.linspace(0, 2*np.pi, 1_000_000)`:
- With a Python loop
- Vectorized

### Exercise 5 (real-world challenge)
You have 1000 samples with 4 features stored as a `(1000, 4)` array. Implement a 70/15/15 train/validation/test split using only NumPy.

In [29]:
# Exercise 1
data = np.array([1.0, 2.0, np.nan, 4.0, 5.0, np.nan, 7.0])
median = np.nanmedian(data)
clean = np.where(np.isnan(data), median, data)
print(f"Median of non-NaN values: {median}")
print(f"Cleaned: {clean}")

Median of non-NaN values: 4.0
Cleaned: [1. 2. 4. 4. 5. 4. 7.]


In [31]:
# Exercise 2
arr = np.arange(100).reshape(10, 10)
np.save('ex2.npy', arr)
loaded = np.load('ex2.npy')
print("Arrays match:", np.array_equal(arr, loaded))
print("Loaded shape:", loaded.shape)

Arrays match: True
Loaded shape: (10, 10)


In [32]:
# Exercise 3
np.random.seed(0)
data = np.random.rand(100, 5)

# Standardize
mean = data.mean(axis=0)
std  = data.std(axis=0)
standardized = (data - mean) / std
print("Standardized means:", standardized.mean(axis=0).round(6))
print("Standardized stds: ", standardized.std(axis=0).round(6))

# Pairwise correlations of columns
corr = np.corrcoef(data.T)   # T because corrcoef expects variables in rows
print("\nCorrelation matrix:")
print(corr.round(3))

# Find max correlation (excluding diagonal)
np.fill_diagonal(corr, 0)   # zero out the diagonal (always 1.0)
max_idx = np.unravel_index(np.abs(corr).argmax(), corr.shape)
print(f"\nHighest correlated columns: {max_idx[0]} and {max_idx[1]} (corr={corr[max_idx]:.4f})")

Standardized means: [ 0.  0.  0.  0. -0.]
Standardized stds:  [1. 1. 1. 1. 1.]

Correlation matrix:
[[ 1.    -0.022  0.217  0.027  0.008]
 [-0.022  1.     0.03   0.159  0.084]
 [ 0.217  0.03   1.     0.06   0.152]
 [ 0.027  0.159  0.06   1.     0.046]
 [ 0.008  0.084  0.152  0.046  1.   ]]

Highest correlated columns: 2 and 0 (corr=0.2168)


In [33]:
# Exercise 4
import time

x = np.linspace(0, 2 * np.pi, 1_000_000)

# Python loop
start = time.time()
y_loop = np.zeros_like(x)
for i in range(len(x)):
    y_loop[i] = np.sin(x[i]) + np.cos(x[i]) * 2
t_loop = time.time() - start

# Vectorized
start = time.time()
y_vec = np.sin(x) + np.cos(x) * 2
t_vec = time.time() - start

print(f"Loop:       {t_loop:.4f}s")
print(f"Vectorized: {t_vec:.4f}s")
print(f"Speedup:    {t_loop/t_vec:.1f}x")

Loop:       16.5606s
Vectorized: 0.0913s
Speedup:    181.5x


In [34]:
# Exercise 5
rng = np.random.default_rng(42)
data = rng.normal(0, 1, (1000, 4))

# Shuffle indices
indices = rng.permutation(1000)

# Compute split points
train_end = int(0.70 * 1000)
val_end   = int(0.85 * 1000)

train_idx = indices[:train_end]
val_idx   = indices[train_end:val_end]
test_idx  = indices[val_end:]

train = data[train_idx]
val   = data[val_idx]
test  = data[test_idx]

print(f"Train: {train.shape} ({len(train)/10:.0f}%)")
print(f"Val:   {val.shape} ({len(val)/10:.0f}%)")
print(f"Test:  {test.shape} ({len(test)/10:.0f}%)")
print(f"Total: {len(train) + len(val) + len(test)}")

Train: (700, 4) (70%)
Val:   (150, 4) (15%)
Test:  (150, 4) (15%)
Total: 1000
